# 01 — Audit Data

Notebook ini akan dikerjakan pada Fase 1 setelah Base dataset tersedia.
Belum ada asumsi mengenai nama kolom, schema, target, atau periode waktu.

In [10]:
from pathlib import Path
import hashlib

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)


def find_project_root(start_path: Path) -> Path:
    """Find the project root containing pyproject.toml."""
    resolved_path = start_path.resolve()

    for candidate in [resolved_path, *resolved_path.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Project root was not found. Open the notebook from the FraudShield repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Base.csv"

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset path : {DATA_PATH}")
print(f"File exists  : {DATA_PATH.exists()}")

assert DATA_PATH.exists(), f"Dataset was not found at: {DATA_PATH}"

Project root : D:\latihan\fraudshield
Dataset path : D:\latihan\fraudshield\data\raw\Base.csv
File exists  : True


In [11]:
def calculate_sha256(file_path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Calculate a SHA-256 checksum without loading the whole file into memory."""
    sha256_hash = hashlib.sha256()

    with file_path.open("rb") as file:
        while chunk := file.read(chunk_size):
            sha256_hash.update(chunk)

    return sha256_hash.hexdigest()


file_size_mib = DATA_PATH.stat().st_size / (1024**2)
file_sha256 = calculate_sha256(DATA_PATH)

print(f"File name    : {DATA_PATH.name}")
print(f"File size    : {file_size_mib:,.2f} MiB")
print(f"SHA-256      : {file_sha256}")

File name    : Base.csv
File size    : 206.56 MiB
SHA-256      : ba7a015e4695399c89da8bf9ffac850be7c23b4666a6cf22af3b3424ecca0957


In [12]:
raw_preview = pd.read_csv(DATA_PATH, nrows=5)

print(f"Preview shape : {raw_preview.shape}")
print(f"Column count  : {len(raw_preview.columns)}")
print("\nActual columns:")

for column_number, column_name in enumerate(raw_preview.columns, start=1):
    print(f"{column_number:02d}. {column_name}")

print("\nPreview data:")
display(raw_preview)

print("\nPreview data types:")
display(raw_preview.dtypes.to_frame(name="dtype"))

Preview shape : (5, 32)
Column count  : 32

Actual columns:
01. fraud_bool
02. income
03. name_email_similarity
04. prev_address_months_count
05. current_address_months_count
06. customer_age
07. days_since_request
08. intended_balcon_amount
09. payment_type
10. zip_count_4w
11. velocity_6h
12. velocity_24h
13. velocity_4w
14. bank_branch_count_8w
15. date_of_birth_distinct_emails_4w
16. employment_status
17. credit_risk_score
18. email_is_free
19. housing_status
20. phone_home_valid
21. phone_mobile_valid
22. bank_months_count
23. has_other_cards
24. proposed_credit_limit
25. foreign_request
26. source
27. session_length_in_minutes
28. device_os
29. keep_alive_session
30. device_distinct_emails_8w
31. device_fraud_count
32. month

Preview data:


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,velocity_6h,velocity_24h,velocity_4w,bank_branch_count_8w,date_of_birth_distinct_emails_4w,employment_status,credit_risk_score,email_is_free,housing_status,phone_home_valid,phone_mobile_valid,bank_months_count,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,1,0.9,0.166828,-1,88,50,0.020925,-1.331345,AA,769,10650.765523,3134.319630,3863.647740,1,6,CA,185,0,BA,1,0,24,0,500.0,0,INTERNET,3.888115,windows,0,1,0,7
1,1,0.9,0.296286,-1,144,50,0.005418,-0.816224,AB,366,534.047319,2670.918292,3124.298166,718,3,CA,259,1,BA,0,0,15,0,1500.0,0,INTERNET,31.798819,windows,0,1,0,7
2,1,0.9,0.044985,-1,132,40,3.108549,-0.755728,AC,870,4048.534263,2893.621498,3159.590679,1,14,CB,177,1,BA,0,1,-1,0,200.0,0,INTERNET,4.728705,other,0,1,0,7
3,1,0.9,0.159511,-1,22,50,0.019079,-1.205124,AB,810,3457.064063,4054.908412,3022.261812,1921,6,CA,110,1,BA,0,1,31,1,200.0,0,INTERNET,2.047904,linux,0,1,0,7
4,1,0.9,0.596414,-1,218,50,0.004441,-0.773276,AB,890,5020.341679,2728.237159,3087.670952,1990,2,CA,295,1,BA,1,0,31,0,1500.0,0,INTERNET,3.775225,macintosh,1,1,0,7



Preview data types:


,dtype
fraud_bool,int64
income,float64
name_email_similarity,float64
prev_address_months_count,int64
current_address_months_count,int64
customer_age,int64
days_since_request,float64
intended_balcon_amount,float64
payment_type,str
zip_count_4w,int64


In [13]:
possible_index_columns = [
    column_name
    for column_name in raw_preview.columns
    if column_name.lower().startswith("unnamed:")
]

print(f"Possible exported index columns: {possible_index_columns}")

Possible exported index columns: []


Pemuatan Dataset Penuh

In [14]:
from time import perf_counter

TARGET_COLUMN = "fraud_bool"
TIME_COLUMN = "month"

load_started_at = perf_counter()

data = pd.read_csv(
    DATA_PATH,
    low_memory=False,
)

load_duration_seconds = perf_counter() - load_started_at
memory_usage_mib = data.memory_usage(deep=True).sum() / (1024**2)

required_columns = {TARGET_COLUMN, TIME_COLUMN}
missing_required_columns = required_columns.difference(data.columns)

assert not missing_required_columns, (
    f"Required columns are missing: {sorted(missing_required_columns)}"
)
assert list(data.columns) == list(raw_preview.columns), (
    "The full dataset schema differs from the preview schema."
)

print(f"Dataset shape        : {data.shape}")
print(f"Memory usage         : {memory_usage_mib:,.2f} MiB")
print(f"Loading duration     : {load_duration_seconds:,.2f} seconds")
print(f"Target column        : {TARGET_COLUMN}")
print(f"Time column          : {TIME_COLUMN}")
print(f"First column         : {data.columns[0]}")
print(f"Last column          : {data.columns[-1]}")
print("\nAll columns:")
print("\n".join(f"{index:02d}. {column}" for index, column in enumerate(data.columns, 1)))

Dataset shape        : (1000000, 32)
Memory usage         : 262.95 MiB
Loading duration     : 3.47 seconds
Target column        : fraud_bool
Time column          : month
First column         : fraud_bool
Last column          : month

All columns:
01. fraud_bool
02. income
03. name_email_similarity
04. prev_address_months_count
05. current_address_months_count
06. customer_age
07. days_since_request
08. intended_balcon_amount
09. payment_type
10. zip_count_4w
11. velocity_6h
12. velocity_24h
13. velocity_4w
14. bank_branch_count_8w
15. date_of_birth_distinct_emails_4w
16. employment_status
17. credit_risk_score
18. email_is_free
19. housing_status
20. phone_home_valid
21. phone_mobile_valid
22. bank_months_count
23. has_other_cards
24. proposed_credit_limit
25. foreign_request
26. source
27. session_length_in_minutes
28. device_os
29. keep_alive_session
30. device_distinct_emails_8w
31. device_fraud_count
32. month


Periksa target dan periode waktu

In [15]:
target_values = sorted(data[TARGET_COLUMN].dropna().unique().tolist())
time_values = sorted(data[TIME_COLUMN].dropna().unique().tolist())

print(f"Target values        : {target_values}")
print(f"Temporal values      : {time_values}")
print(f"Missing target       : {data[TARGET_COLUMN].isna().sum():,}")
print(f"Missing time         : {data[TIME_COLUMN].isna().sum():,}")

target_summary = (
    data[TARGET_COLUMN]
    .value_counts(dropna=False)
    .rename_axis("target_value")
    .reset_index(name="row_count")
)

target_summary["percentage"] = (
    target_summary["row_count"] / len(data) * 100
)

print("\nTarget distribution:")
print(target_summary.to_string(index=False))

temporal_summary = (
    data.groupby(TIME_COLUMN, dropna=False)
    .agg(
        row_count=(TARGET_COLUMN, "size"),
        fraud_count=(TARGET_COLUMN, "sum"),
        fraud_rate=(TARGET_COLUMN, "mean"),
    )
    .reset_index()
)

print("\nTemporal summary:")
print(temporal_summary.to_string(index=False))

Target values        : [0, 1]
Temporal values      : [0, 1, 2, 3, 4, 5, 6, 7]
Missing target       : 0
Missing time         : 0

Target distribution:
 target_value  row_count  percentage
            0     988971     98.8971
            1      11029      1.1029

Temporal summary:
 month  row_count  fraud_count  fraud_rate
     0     132440         1500    0.011326
     1     127620         1198    0.009387
     2     136979         1198    0.008746
     3     150936         1392    0.009222
     4     127691         1452    0.011371
     5     119323         1411    0.011825
     6     108168         1450    0.013405
     7      96843         1428    0.014746


Periksa schema, missing value, dan publikasi

In [16]:
schema_summary = pd.DataFrame(
    {
        "column": data.columns,
        "dtype": data.dtypes.astype(str).values,
        "non_null_count": data.notna().sum().values,
        "null_count": data.isna().sum().values,
        "null_percentage": (
            data.isna().mean().mul(100).round(4).values
        ),
        "unique_count": data.nunique(dropna=False).values,
    }
)

full_duplicate_count = int(data.duplicated().sum())

feature_duplicate_count = int(
    data.drop(columns=[TARGET_COLUMN]).duplicated().sum()
)

print(f"Full duplicate rows  : {full_duplicate_count:,}")
print(f"Feature duplicates   : {feature_duplicate_count:,}")
print(f"Columns with nulls   : {(data.isna().sum() > 0).sum():,}")

print("\nSchema summary:")
print(schema_summary.to_string(index=False))

Full duplicate rows  : 0
Feature duplicates   : 0
Columns with nulls   : 0

Schema summary:
                          column   dtype  non_null_count  null_count  null_percentage  unique_count
                      fraud_bool   int64         1000000           0              0.0             2
                          income float64         1000000           0              0.0             9
           name_email_similarity float64         1000000           0              0.0        998861
       prev_address_months_count   int64         1000000           0              0.0           374
    current_address_months_count   int64         1000000           0              0.0           423
                    customer_age   int64         1000000           0              0.0             9
              days_since_request float64         1000000           0              0.0        989330
          intended_balcon_amount float64         1000000           0              0.0        994971
        

Periksa kandidat sentinel value

In [17]:
SENTINEL_CANDIDATES = (-1, -999, -9999)

numeric_columns = data.select_dtypes(include="number").columns
sentinel_records = []

for column in numeric_columns:
    for sentinel_value in SENTINEL_CANDIDATES:
        sentinel_count = int(data[column].eq(sentinel_value).sum())

        if sentinel_count > 0:
            sentinel_records.append(
                {
                    "column": column,
                    "sentinel_value": sentinel_value,
                    "row_count": sentinel_count,
                    "percentage": round(
                        sentinel_count / len(data) * 100,
                        4,
                    ),
                }
            )

sentinel_summary = pd.DataFrame(sentinel_records)

if sentinel_summary.empty:
    print("No configured sentinel candidates were found.")
else:
    sentinel_summary = sentinel_summary.sort_values(
        ["column", "sentinel_value"]
    )
    print(sentinel_summary.to_string(index=False))

                      column  sentinel_value  row_count  percentage
           bank_months_count              -1     253635     25.3635
           credit_risk_score              -1        488      0.0488
current_address_months_count              -1       4254      0.4254
   device_distinct_emails_8w              -1        359      0.0359
   prev_address_months_count              -1     712920     71.2920
   session_length_in_minutes              -1       2015      0.2015


Tipe data dan kolom yg terpotong

In [18]:
print("Columns 17-28:")

for index in range(16, 28):
    print(f"{index + 1:02d}. {data.columns[index]}")

print("\nData types:")

for column_name, dtype in data.dtypes.items():
    print(f"{column_name:<45} {str(dtype)}")

Columns 17-28:
17. credit_risk_score
18. email_is_free
19. housing_status
20. phone_home_valid
21. phone_mobile_valid
22. bank_months_count
23. has_other_cards
24. proposed_credit_limit
25. foreign_request
26. source
27. session_length_in_minutes
28. device_os

Data types:
fraud_bool                                    int64
income                                        float64
name_email_similarity                         float64
prev_address_months_count                     int64
current_address_months_count                  int64
customer_age                                  int64
days_since_request                            float64
intended_balcon_amount                        float64
payment_type                                  str
zip_count_4w                                  int64
velocity_6h                                   float64
velocity_24h                                  float64
velocity_4w                                   float64
bank_branch_count_8w                  

Nilai kategori, biner, dan kolom konstan

In [19]:
CATEGORICAL_COLUMNS = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
]

BINARY_COLUMNS = [
    "fraud_bool",
    "email_is_free",
    "phone_home_valid",
    "phone_mobile_valid",
    "has_other_cards",
    "foreign_request",
    "keep_alive_session",
]

print("Categorical values:")

for column_name in CATEGORICAL_COLUMNS:
    value_counts = (
        data[column_name]
        .value_counts(dropna=False)
        .sort_index()
        .to_dict()
    )
    print(f"{column_name}: {value_counts}")

print("\nBinary values:")

for column_name in BINARY_COLUMNS:
    unique_values = sorted(
        data[column_name].dropna().unique().tolist()
    )
    print(f"{column_name}: {unique_values}")

constant_columns = [
    column_name
    for column_name in data.columns
    if data[column_name].nunique(dropna=False) == 1
]

print(f"\nConstant columns: {constant_columns}")

Categorical values:
payment_type: {'AA': 258249, 'AB': 370554, 'AC': 252071, 'AD': 118837, 'AE': 289}
employment_status: {'CA': 730252, 'CB': 138288, 'CC': 37758, 'CD': 26522, 'CE': 22693, 'CF': 44034, 'CG': 453}
housing_status: {'BA': 169675, 'BB': 260965, 'BC': 372143, 'BD': 26161, 'BE': 169135, 'BF': 1669, 'BG': 252}
source: {'INTERNET': 992952, 'TELEAPP': 7048}
device_os: {'linux': 332712, 'macintosh': 53826, 'other': 342728, 'windows': 263506, 'x11': 7228}

Binary values:
fraud_bool: [0, 1]
email_is_free: [0, 1]
phone_home_valid: [0, 1]
phone_mobile_valid: [0, 1]
has_other_cards: [0, 1]
foreign_request: [0, 1]
keep_alive_session: [0, 1]

Constant columns: ['device_fraud_count']


Missing VALUE BERDASARKAN DOKUMENTASI

In [20]:
documented_missing_rules = {
    "prev_address_months_count": data[
        "prev_address_months_count"
    ].eq(-1),
    "current_address_months_count": data[
        "current_address_months_count"
    ].eq(-1),
    "bank_months_count": data[
        "bank_months_count"
    ].eq(-1),
    "session_length_in_minutes": data[
        "session_length_in_minutes"
    ].eq(-1),
    "device_distinct_emails_8w": data[
        "device_distinct_emails_8w"
    ].eq(-1),
    "intended_balcon_amount": data[
        "intended_balcon_amount"
    ].lt(0),
}

semantic_missing_records = []

for column_name, missing_mask in documented_missing_rules.items():
    missing_count = int(missing_mask.sum())

    semantic_missing_records.append(
        {
            "column": column_name,
            "missing_count": missing_count,
            "missing_percentage": round(
                missing_count / len(data) * 100,
                4,
            ),
            "observed_min": data[column_name].min(),
            "observed_max": data[column_name].max(),
        }
    )

semantic_missing_summary = pd.DataFrame(
    semantic_missing_records
)

print("Documented semantic missing values:")
print(semantic_missing_summary.to_string(index=False))

special_numeric_columns = [
    "credit_risk_score",
    "velocity_6h",
    "device_fraud_count",
]

special_numeric_summary = data[special_numeric_columns].agg(
    ["min", "max", "nunique"]
).T

print("\nSpecial numeric columns:")
print(special_numeric_summary.to_string())

Documented semantic missing values:
                      column  missing_count  missing_percentage  observed_min  observed_max
   prev_address_months_count         712920             71.2920     -1.000000    383.000000
current_address_months_count           4254              0.4254     -1.000000    428.000000
           bank_months_count         253635             25.3635     -1.000000     32.000000
   session_length_in_minutes           2015              0.2015     -1.000000     85.899143
   device_distinct_emails_8w            359              0.0359     -1.000000      2.000000
      intended_balcon_amount         742523             74.2523    -15.530555    112.956928

Special numeric columns:
                           min           max   nunique
credit_risk_score  -170.000000    389.000000     551.0
velocity_6h        -170.603072  16715.565404  998687.0
device_fraud_count    0.000000      0.000000       1.0


Validasi dataset aktual

In [21]:
import sys
print(sys.executable)

d:\latihan\fraudshield\.venv\Scripts\python.exe


In [22]:
import importlib
import fraudshield.validation as validation

validation = importlib.reload(validation)
validate_base_dataset = validation.validate_base_dataset

validated_data = validate_base_dataset(data)
print("Raw Base dataset schema validation passed.")

SchemaErrors: {
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": "BAF Base raw-data schema",
                "column": "proposed_credit_limit",
                "check": "in_range(200, 2000)",
                "error": "Column 'proposed_credit_limit' failed element-wise validator number 1: in_range(200, 2000) failure cases: 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 190.0, 2100.0, 2100.0, 190.0, 190.0, 190.0, 190.0, 2100.0, 2100.0, 2100.0, 2100.0, 2100.0, 2100.0, 2100.0, 2100.0, 2100.0, 190.0, 1